# Week 79, Day 5: Cryptography basics

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w079d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import base64, hashlib

secret = b"meet at noon"
encoded = base64.b64encode(secret)
print(encoded, base64.b64decode(encoded))

digest = hashlib.sha256(secret).hexdigest()
print(len(digest), digest[:16])
print(hashlib.sha256(b"meet at noon!").hexdigest()[:16])

In [ ]:
import hashlib, hmac

key = b"server-side secret key"
message = b"user=7&role=reader"
tag = hmac.new(key, message, hashlib.sha256).hexdigest()

def check(message, tag):
    expected = hmac.new(key, message, hashlib.sha256).hexdigest()
    return hmac.compare_digest(expected, tag)

print(check(message, tag), check(b"user=7&role=admin", tag))
print(hashlib.md5(b"x").digest_size, hashlib.sha256(b"x").digest_size, hashlib.blake2b(b"x", digest_size=32).digest_size)

In [ ]:
# A toy Diffie-Hellman with tiny numbers (real ones use 2048-bit groups or elliptic curves).
p, g = 23, 5                       # public: a prime and a generator
alice_secret, bob_secret = 6, 15   # each keeps theirs private
A = pow(g, alice_secret, p)        # Alice sends A
B = pow(g, bob_secret, p)          # Bob sends B
print(A, B, pow(B, alice_secret, p), pow(A, bob_secret, p))

# A toy RSA key from two small primes.
p, q, e = 61, 53, 17
n, phi = p * q, (p - 1) * (q - 1)
d = pow(e, -1, phi)                # the private exponent
message = 65
cipher = pow(message, e, n)        # anyone can encrypt with (n, e)
print(n, d, cipher, pow(cipher, d, n))

> **Key idea.** Encoding (Base64) hides nothing; hashes (SHA-256, SHA-3, BLAKE2; not MD5 or SHA-1) give integrity and identity; MACs (HMAC, never sha256(key + message), which allows length extension) add authenticity with a shared key, checked with compare_digest. Passwords need slow, salted hashes (Argon2id, scrypt, bcrypt, or PBKDF2 with around 600,000 iterations), stored as self-describing strings and re-hashed when the work factor rises; the same functions derive keys from passwords. Symmetric ciphers need a key and a never-repeated nonce (reuse leaks the XOR of plaintexts) and must be authenticated (AES-GCM, ChaCha20-Poly1305, Fernet), verifying the tag, and any associated data, before using the plaintext. Public-key pairs encrypt to or verify for anyone with the public key (RSA, Ed25519); Diffie-Hellman agrees secrets in the open; TLS combines certificates, key exchange and authenticated encryption; never disable certificate verification. Never design or implement cryptography for real use: use the cryptography library's high-level APIs.

### Exercise: A signed cookie

Write `sign(value, key)`: `VALUE.TAG` where TAG is the URL-safe Base64 (without `=` padding) of the HMAC-SHA256 of the value's UTF-8 bytes with `key` (bytes). Then `unsign(cookie, key)`: the value if the tag is correct (split at the **last** `.`, recompute and compare with `hmac.compare_digest`), else raise `ValueError("bad signature")` (also for a cookie without a `.`).

In [ ]:
import base64
import hashlib
import hmac

def sign(value, key):
    ___

def unsign(cookie, key):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cookie():\n    \"tamper-evident\"\n    key = b\"k\" * 32\n    c = sign(\"user=7.role=reader\", key)\n    assert c.startswith(\"user=7.role=reader.\") and \"=\" not in c.rsplit(\".\", 1)[1] and len(c.rsplit(\".\", 1)[1]) == 43\n    assert unsign(c, key) == \"user=7.role=reader\"\n    for bad in [c.replace(\"reader\", \"admin\"), c[:-1] + (\"A\" if c[-1] != \"A\" else \"B\"), \"nodot\", sign(\"user=7.role=reader\", b\"other key\")]:\n        try:\n            unsign(bad, key)\n        except ValueError as e:\n            assert str(e) == \"bad signature\"\n        else:\n            raise AssertionError(bad)\n    assert \"compare_digest\" in source")

### Exercise: PBKDF2 from HMAC

Write `pbkdf2_sha256(password, salt, iterations, dklen)` from the description in the lesson, using only `hmac` and `hashlib.sha256` (no `hashlib.pbkdf2_hmac`, which the browser doesn't have): `password` and `salt` are bytes; return `dklen` bytes. Each block is the XOR of `iterations` HMAC results (XOR two byte strings by converting them to integers with `int.from_bytes(..., "big")`, or byte by byte). Your function must match the published test vectors (from RFC 7914), which the checks use. Tip: `h = hmac.new(password, digestmod=hashlib.sha256)` then `h.copy()` for each HMAC saves re-processing the key.

In [ ]:
import hashlib
import hmac

def pbkdf2_sha256(password, salt, iterations, dklen):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_vectors():\n    \"RFC 7914 and widely published vectors\"\n    assert pbkdf2_sha256(b\"password\", b\"salt\", 1, 32).hex() == \"120fb6cffcf8b32c43e7225256c4f837a86548c92ccc35480805987cb70be17b\"\n    assert pbkdf2_sha256(b\"password\", b\"salt\", 2, 32).hex() == \"ae4d0c95af6b46d32d0adff928f06dd02a303f8ef3c251dfd6e2d85a95474c43\"\n    assert pbkdf2_sha256(b\"password\", b\"salt\", 4096, 32).hex() == \"c5e478d59288c841aa530db6845c4c8d962893a001ce4e11a4963873aa98134a\"\n    long = pbkdf2_sha256(b\"passwordPASSWORDpassword\", b\"saltSALTsaltSALTsaltSALTsaltSALTsalt\", 4096, 40)\n    assert long.hex() == \"348c89dbcbd32b2f32d814b8116e84cf2b17347ebc1800181c4e2a1fb8dd53e1c635518c7dac47e9\"\n    assert len(pbkdf2_sha256(b\"p\", b\"s\", 3, 5)) == 5\n    assert \"pbkdf2_hmac\" not in source")

### Exercise: Storing passwords properly

Using the given `pbkdf2_sha256`, write `hash_password(password, iterations=2000, salt=None)`: a string `pbkdf2_sha256$ITERATIONS$SALT$HASH`, where SALT is 16 random bytes from `secrets.token_bytes` (unless `salt` is given) and both SALT and HASH (32 bytes) are standard Base64 (`base64.b64encode`, decoded to text). Then `check_password(password, stored)`: `True` if it matches (compare with `hmac.compare_digest`), `False` for a wrong password or a string in any other format. And `needs_rehash(stored, iterations)`: `True` if the stored hash used fewer iterations than `iterations`. (A real system would use hundreds of thousands of iterations; the checks use few to stay quick.)

In [ ]:
import base64
import hashlib
import hmac
import secrets

def pbkdf2_sha256(password, salt, iterations, dklen):
    base = hmac.new(password, digestmod=hashlib.sha256)
    out, block = b"", 1
    while len(out) < dklen:
        h = base.copy()
        h.update(salt + block.to_bytes(4, "big"))
        u = h.digest()
        total = int.from_bytes(u, "big")
        for _ in range(iterations - 1):
            h = base.copy()
            h.update(u)
            u = h.digest()
            total ^= int.from_bytes(u, "big")
        out += total.to_bytes(32, "big")
        block += 1
    return out[:dklen]

def hash_password(password, iterations=2000, salt=None):
    ___

def check_password(password, stored):
    ___

def needs_rehash(stored, iterations):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_store():\n    \"salted, self-describing, checkable\"\n    a, b = hash_password(\"correct horse\"), hash_password(\"correct horse\")\n    assert a != b and a.startswith(\"pbkdf2_sha256$2000$\") and len(a.split(\"$\")) == 4\n    assert check_password(\"correct horse\", a) and check_password(\"correct horse\", b) and not check_password(\"Correct horse\", a)\n    fixed = hash_password(\"pw\", 1, salt=b\"salt\")\n    assert fixed == \"pbkdf2_sha256$1$c2FsdA==$b0rYx47DZcBg5kjraU7kDepYSEsDcfvWFxWsRBC3OAo=\"\n    assert check_password(\"pw\", fixed)\n\ndef test_formats_and_rehash():\n    \"bad formats and work factors\"\n    for bad in [\"\", \"md5$1$x$y\", \"pbkdf2_sha256$x$c2FsdA==$abc\", \"pbkdf2_sha256$1$c2FsdA==\", \"pbkdf2_sha256$1$!!$!!\"]:\n        assert check_password(\"pw\", bad) is False, bad\n    stored = hash_password(\"pw\", 1000)\n    assert needs_rehash(stored, 2000) and not needs_rehash(stored, 1000)\n    assert \"compare_digest\" in source")

### Exercise: Authenticated encryption, by hand

For learning only (in real code use `cryptography`'s `AESGCM` or `Fernet`), build a small authenticated cipher from SHA-256 and HMAC. `keystream(key, nonce, n)`: the first `n` bytes of `sha256(key + nonce + counter)` blocks for counter `0, 1, 2...` as 8 big-endian bytes, joined. `seal(key, nonce, plaintext, aad=b"")`: derive an encryption key `HMAC(key, b"enc")` and a MAC key `HMAC(key, b"mac")` (SHA-256 digests), XOR the plaintext with the keystream from the encryption key, then compute the tag `HMAC(mac_key, nonce + len(aad) as 8 big-endian bytes + aad + ciphertext)`; return `nonce + ciphertext + tag` (the nonce is 12 bytes, the tag 32). `open_sealed(key, sealed, aad=b"")`: split it, recompute and check the tag with `hmac.compare_digest` **before** decrypting, raising `ValueError("message was tampered with")` if it doesn't match (or the input is too short), and return the plaintext.

In [ ]:
import hashlib
import hmac

def keystream(key, nonce, n):
    ___

def seal(key, nonce, plaintext, aad=b""):
    ___

def open_sealed(key, sealed, aad=b""):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_roundtrip():\n    \"encrypts and decrypts\"\n    import hashlib\n    key, nonce = b\"k\" * 32, b\"n\" * 12\n    ks = keystream(b\"key\", b\"nonce\", 40)\n    assert ks[:32] == hashlib.sha256(b\"key\" + b\"nonce\" + (0).to_bytes(8, \"big\")).digest() and len(ks) == 40\n    sealed = seal(key, nonce, b\"pay Ada \\xc2\\xa310\", b\"invoice-7\")\n    assert sealed[:12] == nonce and len(sealed) == 12 + 12 + 32 and b\"Ada\" not in sealed\n    assert open_sealed(key, sealed, b\"invoice-7\") == b\"pay Ada \\xc2\\xa310\"\n    assert seal(key, nonce, b\"\")[12:-32] == b\"\"\n\ndef test_tamper():\n    \"any change is refused\"\n    key, nonce = b\"k\" * 32, b\"n\" * 12\n    sealed = bytearray(seal(key, nonce, b\"pay Ada 10\", b\"invoice-7\"))\n    for i in (0, 12, 20, len(sealed) - 1):\n        bad = bytearray(sealed)\n        bad[i] ^= 1\n        try:\n            open_sealed(key, bytes(bad), b\"invoice-7\")\n        except ValueError as e:\n            assert str(e) == \"message was tampered with\"\n        else:\n            raise AssertionError(f\"byte {i}\")\n    for args in ((b\"x\" * 32, bytes(sealed), b\"invoice-7\"), (key, bytes(sealed), b\"invoice-8\"), (key, b\"short\", b\"\")):\n        try:\n            open_sealed(*args)\n        except ValueError:\n            pass\n        else:\n            raise AssertionError(\"must refuse\")")

### Exercise: What a reused nonce gives away

Two messages were encrypted with the same key **and the same nonce**, so with the same keystream: `c1 = p1 XOR ks` and `c2 = p2 XOR ks`. Write `xor(a, b)` for two byte strings (as long as the shorter one). Then `recover(c1, c2, known_p1)`: given both ciphertexts and the first plaintext (which the attacker guessed, say a standard header), return the second plaintext, as far as both are long; no key needed. Then `bit_flip(ciphertext, known_plaintext, wanted)`: a forged ciphertext that decrypts to `wanted` (the same length as the known plaintext), showing why unauthenticated encryption can't be trusted.

In [ ]:
def xor(a, b):
    ___

def recover(c1, c2, known_p1):
    ___

def bit_flip(ciphertext, known_plaintext, wanted):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_reuse():\n    \"two-time pad\"\n    import hashlib\n    ks = b\"\".join(hashlib.sha256(b\"secret\" + bytes([i])).digest() for i in range(2))\n    p1, p2 = b\"From: payroll@acme.example\", b\"Bonus for Ada: 4,000 pounds\"\n    c1, c2 = xor(p1, ks), xor(p2, ks)\n    assert xor(b\"abc\", b\"\\x01\\x02\") == b\"``\" and xor(xor(p1, ks), ks) == p1\n    assert recover(c1, c2, p1) == p2[:len(p1)]\n\ndef test_flip():\n    \"forging without the key\"\n    import hashlib\n    ks = hashlib.sha256(b\"k\").digest()\n    c = xor(b\"pay Ada 10 pounds\", ks)\n    forged = bit_flip(c, b\"pay Ada 10 pounds\", b\"pay Bob 90 pounds\")\n    assert xor(forged, ks) == b\"pay Bob 90 pounds\"")

### Exercise: Signatures with a toy RSA key

With **tiny** numbers, so you can see the maths (real keys have primes of over 1000 bits each and use padding schemes such as PSS): write `make_keys(p, q, e)`: `((n, e), (n, d))`, with `n = p*q` and `d` the inverse of `e` modulo `(p-1)*(q-1)` (use `pow(e, -1, phi)`). Then `digest_int(message, n)`: the SHA-256 of the message bytes as a big-endian integer, modulo `n`. `sign(message, private)`: `pow(digest_int(message, n), d, n)`. `verify(message, signature, public)`: `True` if `pow(signature, e, n)` equals `digest_int(message, n)`.

In [ ]:
import hashlib

def make_keys(p, q, e):
    ___

def digest_int(message, n):
    ___

def sign(message, private):
    ___

def verify(message, signature, public):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_rsa():\n    \"sign with private, verify with public\"\n    public, private = make_keys(61, 53, 17)\n    assert public == (3233, 17) and private == (3233, 2753)\n    s = sign(b\"release 2.0 is genuine\", private)\n    assert 0 <= s < 3233 and verify(b\"release 2.0 is genuine\", s, public)\n    assert not verify(b\"release 2.0 is genuine\", (s + 1) % 3233, public)\n    pub2, priv2 = make_keys(1009, 1013, 65537)\n    msg = b\"deploy build 4521\"\n    assert verify(msg, sign(msg, priv2), pub2) and not verify(b\"deploy build 4522\", sign(msg, priv2), pub2)")

### Bug hunt: Passwords that gave each other away

Passwords are hashed without a salt, so two users with the same password have the same hash (and one cracked hash cracks both): this prints `same hash: True`. Add a random 16-byte salt per user (stored alongside, as `salt + digest`), so it prints `same hash: False` while each hash still checks out: `checks: True True`.

In [ ]:
import hashlib, secrets

def store(password):
    return hashlib.sha256(password.encode()).digest()

def check(password, stored):
    return hashlib.sha256(password.encode()).digest() == stored

ada, bob = store("summer2026"), store("summer2026")
print("same hash:", ada == bob)
print("checks:", check("summer2026", ada), check("summer2026", bob))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a salt each\"\n    expect_output(\"same hash: False\\nchecks: True True\")")

### Bug hunt: A tag that only had to appear

The check accepts any token that **contains** the right tag, so a token with extra characters glued on passes, and this prints `forged accepted: True`. Compare the whole tag exactly, in constant time, so it prints `forged accepted: False`.

In [ ]:
import hashlib, hmac

KEY = b"server key"

def tag(message):
    return hmac.new(KEY, message, hashlib.sha256).hexdigest()

def check(message, token):
    return tag(message) in token

good = tag(b"user=7")
print("forged accepted:", check(b"user=7", "ZZ" + good + "&admin=1"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"compare_digest\"\n    expect_output(\"forged accepted: False\")\n    assert \"compare_digest\" in source")

### Bug hunt: A nonce that never changed

Every message is encrypted with the same nonce, so equal messages give equal ciphertexts (and worse, the keystream repeats), and this prints `repeats visible: True`. Use a fresh random 12-byte nonce for each message (`secrets.token_bytes(12)`), sent in front of the ciphertext, so it prints `repeats visible: False`.

In [ ]:
import hashlib, secrets

KEY = b"k" * 32
NONCE = b"\x00" * 12

def encrypt(message):
    ks = hashlib.sha256(KEY + NONCE).digest()
    return NONCE + bytes(a ^ b for a, b in zip(message, ks))

print("repeats visible:", encrypt(b"attack at dawn") == encrypt(b"attack at dawn"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a fresh nonce each time\"\n    expect_output(\"repeats visible: False\")")

### Puzzle: Opening an authenticated message

Arrange a function that decrypts only after checking the tag. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # "encoding", "hash" or "MAC": what Base64 is
q = ___   # hashlib.sha256(b"").digest_size
r = ___   # True or False: MD5 is fine for checking a download's authenticity
s = ___   # pow(3, -1, 7)
t = ___   # what must never repeat with the same key: "salt" or "nonce"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"encoding\" and q == 32 and r is False, \"p, q and r\"\n    assert s == 5 and t == \"nonce\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a notes vault

Write a class `Vault(password, salt, iterations=1000)` that keeps encrypted notes, using the given `pbkdf2_sha256`, `seal` and `open_sealed`. The vault's key is `pbkdf2_sha256(password_bytes, salt, iterations, 32)`. It also stores a **check value**: `seal(key, 12 zero bytes, b"vault-check", aad=b"check")`, so a wrong password can be detected without trying a note. `unlock(password)` (a class method taking the stored state) is described below. Methods: `put(name, text)` seals the note's UTF-8 text with a fresh `secrets.token_bytes(12)` nonce and the note's **name as associated data** (so a note's ciphertext can't be moved under another name), storing the result in `self.notes[name]`; `get(name)` returns the text (`KeyError` for an unknown name; `ValueError("note was tampered with")` if opening fails); `state()` returns a dict `{"salt": salt, "iterations": n, "check": check_bytes, "notes": {name: sealed_bytes}}` (copies); `Vault.unlock(state, password)` builds a vault from a state, raising `ValueError("wrong password")` if the check value doesn't open; and `change_password(new_password, new_salt)` re-encrypts every note under the new key (with fresh nonces), replacing the salt and check value.

In [ ]:
import hashlib
import hmac
import secrets

def pbkdf2_sha256(password, salt, iterations, dklen):
    base = hmac.new(password, digestmod=hashlib.sha256)
    out, block = b"", 1
    while len(out) < dklen:
        h = base.copy()
        h.update(salt + block.to_bytes(4, "big"))
        u = h.digest()
        total = int.from_bytes(u, "big")
        for _ in range(iterations - 1):
            h = base.copy()
            h.update(u)
            u = h.digest()
            total ^= int.from_bytes(u, "big")
        out += total.to_bytes(32, "big")
        block += 1
    return out[:dklen]

def keystream(key, nonce, n):
    out, counter = b"", 0
    while len(out) < n:
        out += hashlib.sha256(key + nonce + counter.to_bytes(8, "big")).digest()
        counter += 1
    return out[:n]

def _keys(key):
    return hmac.new(key, b"enc", hashlib.sha256).digest(), hmac.new(key, b"mac", hashlib.sha256).digest()

def _tag(mac_key, nonce, aad, ciphertext):
    return hmac.new(mac_key, nonce + len(aad).to_bytes(8, "big") + aad + ciphertext, hashlib.sha256).digest()

def seal(key, nonce, plaintext, aad=b""):
    enc_key, mac_key = _keys(key)
    ciphertext = bytes(a ^ b for a, b in zip(plaintext, keystream(enc_key, nonce, len(plaintext))))
    return nonce + ciphertext + _tag(mac_key, nonce, aad, ciphertext)

def open_sealed(key, sealed, aad=b""):
    if len(sealed) < 44:
        raise ValueError("message was tampered with")
    nonce, ciphertext, tag = sealed[:12], sealed[12:-32], sealed[-32:]
    enc_key, mac_key = _keys(key)
    if not hmac.compare_digest(tag, _tag(mac_key, nonce, aad, ciphertext)):
        raise ValueError("message was tampered with")
    return bytes(a ^ b for a, b in zip(ciphertext, keystream(enc_key, nonce, len(ciphertext))))

class Vault:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_notes():\n    \"put, get, and nothing readable at rest\"\n    v = Vault(\"correct horse\", b\"salt-1234567890!\", 500)\n    v.put(\"bank\", \"PIN 4821\")\n    v.put(\"wifi\", \"router: hunter22\")\n    assert v.get(\"bank\") == \"PIN 4821\" and v.get(\"wifi\") == \"router: hunter22\"\n    state = v.state()\n    assert set(state) == {\"salt\", \"iterations\", \"check\", \"notes\"} and state[\"iterations\"] == 500\n    assert b\"4821\" not in b\"\".join(state[\"notes\"].values()) and b\"hunter22\" not in b\"\".join(state[\"notes\"].values())\n    v.put(\"bank\", \"PIN 4821\")\n    assert v.state()[\"notes\"][\"bank\"] != state[\"notes\"][\"bank\"], \"a fresh nonce every time\"\n    try:\n        v.get(\"email\")\n    except KeyError:\n        pass\n    else:\n        raise AssertionError(\"unknown note\")\n\ndef test_unlock():\n    \"right and wrong passwords\"\n    v = Vault(\"correct horse\", b\"salt-1234567890!\", 500)\n    v.put(\"bank\", \"PIN 4821\")\n    state = v.state()\n    again = Vault.unlock(state, \"correct horse\")\n    assert again.get(\"bank\") == \"PIN 4821\"\n    try:\n        Vault.unlock(state, \"Correct horse\")\n    except ValueError as e:\n        assert str(e) == \"wrong password\"\n    else:\n        raise AssertionError(\"wrong password accepted\")\n\ndef test_tampering():\n    \"swapped and changed notes are refused\"\n    v = Vault(\"pw\", b\"s\" * 16, 200)\n    v.put(\"bank\", \"PIN 4821\")\n    v.put(\"wifi\", \"hunter22\")\n    state = v.state()\n    state[\"notes\"][\"wifi\"] = state[\"notes\"][\"bank\"]\n    w = Vault.unlock(state, \"pw\")\n    try:\n        w.get(\"wifi\")\n    except ValueError as e:\n        assert str(e) == \"note was tampered with\"\n    else:\n        raise AssertionError(\"a note moved under another name must be refused\")\n    state = v.state()\n    blob = bytearray(state[\"notes\"][\"bank\"])\n    blob[13] ^= 4\n    state[\"notes\"][\"bank\"] = bytes(blob)\n    try:\n        Vault.unlock(state, \"pw\").get(\"bank\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"a changed byte must be refused\")\n\ndef test_change_password():\n    \"re-encrypted under a new key\"\n    v = Vault(\"old\", b\"a\" * 16, 200)\n    v.put(\"bank\", \"PIN 4821\")\n    before = v.state()[\"notes\"][\"bank\"]\n    v.change_password(\"new\", b\"b\" * 16)\n    state = v.state()\n    assert state[\"salt\"] == b\"b\" * 16 and state[\"notes\"][\"bank\"] != before\n    assert Vault.unlock(state, \"new\").get(\"bank\") == \"PIN 4821\"\n    try:\n        Vault.unlock(state, \"old\")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"the old password must stop working\")")

## Warm-up

Base64 is an **encoding**: anyone can reverse it, so it hides nothing (it's a way to put bytes into text). A **hash** can't be reversed, always has the same length, and changes completely when the input changes by one character. Mixing these two up ("the password is base64-encoded, so it's safe") is a classic mistake.

## What each tool promises

| Tool | What it gives | Key? | Python |
|---|---|---|---|
| **Encoding** (Base64, hex) | nothing secret; bytes as text | none | `base64`, `bytes.hex()` |
| **Hash** (SHA-256) | a fingerprint: **integrity** against accidents, identity of content | none | `hashlib.sha256` |
| **MAC** (HMAC-SHA256) | **integrity and authenticity**: only someone with the key could have made it | one shared secret key | `hmac.new` |
| **Password hash / KDF** (PBKDF2, scrypt, Argon2) | a slow, salted fingerprint of a password, or a key derived from one | a password and a salt | `hashlib.pbkdf2_hmac`, `hashlib.scrypt` |
| **Symmetric encryption** (AES-GCM, ChaCha20-Poly1305) | **confidentiality** (and, in these "authenticated" modes, integrity) | one shared secret key | the `cryptography` library |
| **Public-key encryption and key exchange** (RSA, elliptic-curve Diffie-Hellman) | agreeing a secret, or sending one, to someone without a shared key | a key pair: public and private | `cryptography` |
| **Digital signature** (Ed25519, RSA-PSS, ECDSA) | integrity, authenticity, and **non-repudiation**: anyone can check it with the public key; only the private key could make it | a key pair | `cryptography` |

The **first rule**: don't design cryptography, and don't implement it for real use either. Use a well-reviewed library (in Python, `cryptography`, or `hashlib` and `hmac` from the standard library), with its recommended high-level API. Today's hand-made versions are for understanding how the pieces work; each one says what to use instead.

## Hashes and MACs

A **cryptographic hash** maps any input to a fixed-size digest (32 bytes for SHA-256) so that it's infeasible to find an input with a given digest (**preimage resistance**) or two inputs with the same digest (**collision resistance**). Uses: checking downloads (Thursday), identifying content (Week 78's duplicates), and as a building block for everything else. **MD5** and **SHA-1** have practical collision attacks and must not be used for security; SHA-256, SHA-512, SHA-3 and BLAKE2 are fine.

A hash alone proves nothing about **who** made it: an attacker who changes a message just recomputes its hash. A **MAC** mixes in a secret key, so only key holders can produce a valid tag. Use **HMAC**, not `sha256(key + message)`: that naive construction is open to a **length-extension** attack, where anyone who sees one valid tag can compute a valid tag for the message with extra data appended, without knowing the key. HMAC's design (hashing twice, with the key mixed in differently each time) prevents that. And always check tags with `hmac.compare_digest` (Monday).

## Passwords and keys

**Passwords need slow hashes** (Week 77). A fast hash like SHA-256 can be computed billions of times a second on a graphics card, so a stolen table of SHA-256 password hashes falls to guessing almost at once. Password hashing functions are **deliberately slow** and **salted**:

- a **salt** is a random value stored with each hash, so the same password gives different hashes for different users, and precomputed tables are useless;
- a **work factor** (iterations, or memory in scrypt and Argon2) makes each guess expensive; raise it as computers get faster, and re-hash a user's password with the new setting the next time they log in.

**PBKDF2** (from the standard RFC 8018) is the simplest: it applies HMAC with the password as the key, thousands of times, and XORs the results together. Current guidance (OWASP's) for PBKDF2-HMAC-SHA256 is around 600,000 iterations; **Argon2id** is preferred for new systems, and **scrypt** and **bcrypt** are also fine. The same functions **derive keys** from passwords (a **KDF**: key derivation function), which is how a password-protected file gets an encryption key.

```
PBKDF2(P, S, c, dkLen):
    for each block number i = 1, 2, ... (until there are dkLen bytes):
        U1 = HMAC(P, S + i as 4 big-endian bytes)
        U2 = HMAC(P, U1), ..., Uc = HMAC(P, Uc-1)
        block i = U1 XOR U2 XOR ... XOR Uc
    return the blocks joined, cut to dkLen bytes
```

Stored password hashes are usually written as one self-describing string, such as `pbkdf2_sha256$600000$SALT$HASH`, so the algorithm and work factor can change over time.

## Encryption and signatures

**Symmetric encryption** turns plaintext into ciphertext with a key; the same key turns it back. Modern ciphers such as AES and ChaCha20 are used in modes that need a **nonce** ("number used once"): a value that must **never repeat** with the same key. Many modes work by making a **keystream** from the key and nonce and XORing it with the message; reuse a nonce, and two ciphertexts XORed together give the two plaintexts XORed together, from which an attacker who knows (or guesses) one recovers the other.

Encryption alone doesn't stop **tampering**: flipping a bit of a stream-cipher ciphertext flips the same bit of the plaintext, so `pay £10` can become `pay £90` without the key. Use **authenticated encryption** (AEAD: AES-GCM, ChaCha20-Poly1305, or Fernet, which uses AES with HMAC): it adds a tag, and decryption **refuses** anything that was changed. It can also authenticate **associated data** that isn't secret but must not be swapped (a record's id, a header). The rule: **check the tag before doing anything with the plaintext.**

**Public-key cryptography** uses a **key pair**. With RSA (whose security rests on how hard it is to factor a product of two large primes), anyone can encrypt with the **public** key, and only the **private** key decrypts; or the private key **signs** and anyone can verify with the public key. Elliptic-curve methods (Ed25519 for signatures, X25519 for key exchange) do the same jobs with much smaller keys. **Diffie-Hellman** key exchange lets two parties agree a shared secret over an open channel, which then keys fast symmetric encryption.

**TLS** (the S in HTTPS) puts it all together: the server proves who it is with a **certificate** (its public key, signed by a certificate authority your system trusts) and a signature; both sides agree fresh keys with elliptic-curve Diffie-Hellman; then everything is sent with authenticated symmetric encryption. In Python, `ssl` and every HTTP library do this for you; the one thing to remember is **never turn off certificate verification** (`verify=False`), because then anyone in the middle can pretend to be the server.

### On your own computer

In real code, use the `cryptography` package (`pip install cryptography`): `Fernet` for "encrypt this with a key" (`Fernet.generate_key()`, then `f.encrypt(data)` and `f.decrypt(token)`, which refuses tampered tokens), `AESGCM` when you need associated data, `Ed25519PrivateKey` for signatures, and `Scrypt` or `hashlib.scrypt` for keys from passwords. For password storage in web apps, `argon2-cffi` (`PasswordHasher().hash(pw)` and `.verify(hash, pw)`). Keep keys in a secret manager (Wednesday), never next to the data they protect.

## Practice

## Quiz

## Daily challenge

## Before you go

Open a site you use in your browser, click the padlock, and look at its certificate: who it was issued to, who signed it, and when it expires. Tomorrow is the Week 79 **Forge**: a security review of a small web service, finding and fixing everything this week covered.